# Normalize

The formula sqrt((x₂-x₁)² + (y₂-y₁)²) computes the straight-line distance between them.

In [32]:
import math

# PROBLEMA: ESCALAS DIFERENTES
datos = [
    {"edad": 25, "ingresos": 50000},
    {"edad": 35, "ingresos": 75000},
    {"edad": 45, "ingresos": 100000}
]

item_01 = datos[0]
item_02 = datos[1]

distance = math.sqrt((item_02.get("edad") - item_01.get("edad")) ** 2 + (item_02.get("ingresos") - item_01.get("ingresos")) ** 2)
print(f" distance {distance}")

 distance 25000.00199999992


## Min-Max Normalization (0-1)
- Escala valores al rango [0, 1]. Simple pero sensible a outliers.
- X_norm = (X - min) / (max - min)

In [33]:
datos = [
    {"id": 1, "edad": 20},
    {"id": 2, "edad": 30},
    {"id": 3, "edad": 40},
    {"id": 4, "edad": 50}
]


def max_min_normalization(items: list, key: str) -> list:
    raw_values = [r.get(key) for r in items]
    v_max = max(raw_values)
    v_min = min(raw_values)
    range = v_max - v_min

    if range == 0:
        return [0 for _ in items]

    result = []
    for r in items:
        value = r.get(key)
        value_normalized = (value - v_min) / (v_max - v_min)
        print(f"value: {value} - {v_min} | {(value - v_min)} | {(v_max - v_min)} | ")
        result.append(value_normalized)

    return result

age_normalized = max_min_normalization(datos, "edad")
for idx, age in enumerate(age_normalized):
    print(f"i: {idx} | {datos[idx].get("edad")} | {age}")
    

value: 20 - 20 | 0 | 30 | 
value: 30 - 20 | 10 | 30 | 
value: 40 - 20 | 20 | 30 | 
value: 50 - 20 | 30 | 30 | 
i: 0 | 20 | 0.0
i: 1 | 30 | 0.3333333333333333
i: 2 | 40 | 0.6666666666666666
i: 3 | 50 | 1.0


## Min-Max Normalization (0-1) Multiple fields

>
> Escala valores al rango [0, 1]. Simple pero sensible a outliers.
>

In [34]:
datos_multi = [
    {"edad": 25, "ingresos": 50000},
    {"edad": 35, "ingresos": 75000},
    {"edad": 45, "ingresos": 100000}
]


def normalize_multitple(items: list, fields: list):
    """return normalized"""
    
    result = []
    
    for f in fields:
        values = [r[f] for r in items]
        min_val = min(values)
        max_val = max(values)
        range = max_val - min_val if max_val != min_val else 1
        
        for idx, i in enumerate(items):
            print(f"idx {idx} | i {i}")
            if idx == len(result):
                result.append({})
            normalized = (i[f] - min_val) / range
            result[idx][f"{f}_normalized"] = normalized
        
        print(result)
            
normalize_multitple(datos_multi, ["edad", "ingresos"])

idx 0 | i {'edad': 25, 'ingresos': 50000}
idx 1 | i {'edad': 35, 'ingresos': 75000}
idx 2 | i {'edad': 45, 'ingresos': 100000}
[{'edad_normalized': 0.0}, {'edad_normalized': 0.5}, {'edad_normalized': 1.0}]
idx 0 | i {'edad': 25, 'ingresos': 50000}
idx 1 | i {'edad': 35, 'ingresos': 75000}
idx 2 | i {'edad': 45, 'ingresos': 100000}
[{'edad_normalized': 0.0, 'ingresos_normalized': 0.0}, {'edad_normalized': 0.5, 'ingresos_normalized': 0.5}, {'edad_normalized': 1.0, 'ingresos_normalized': 1.0}]


## Estandarizacion Z-Score

> Escala datos a media 0 y desv.estandar 1. Mejor para distribuciones normales.

- Z-SCORE STANDARDIZATION
- Z = (X - media) / desv_estandar

In [35]:
import math
from statistics import mean, stdev

datos = [
    {"id": 1, "edad": 20},
    {"id": 2, "edad": 30},
    {"id": 3, "edad": 40},
    {"id": 4, "edad": 50}
]


def norm_z(items: list, key: str) -> list:
    """using z"""

    values = [r[key] for r in items]
    values_mean = mean(values)
    # print(f"key {key} | {values_mean}")

    if len(values) < 2:
        return [0 for _ in items]

    desv = stdev(values)
    if desv == 0:
        return [0 for _ in items]

    result = []
    for i in items:
        value = (i[key] - values_mean) / desv
        result.append(value)

    return result


print(f"="*60)
print("nomalization with z-score")
print(f"="*60)
z_values = norm_z(datos, "edad")
for z, j in zip(z_values, datos):
    print(f"z: {z:+.4f}  | j: {j["edad"]}")

print("")
print("infor about with z-score")
z_values_mean = mean(z_values)
z_values_dev = stdev(z_values)
print(f"mean: {z_values_mean} | desv {z_values_dev}")

print(f"="*60)
print("z_values outliers")
print(f"="*60)


def find_outliers(items: list, key: str, umbral: 3):

    z_values = norm_z(items, key)
    outliers = []

    for idx, z in enumerate(z_values):
        if abs(z) > umbral:
            outliers.append((idx, z, items[idx]))

    return outliers


datos_outliers = [
    {"id": 1, "ingresos": 50000},
    {"id": 2, "ingresos": 55000},
    {"id": 3, "ingresos": 52000},
    {"id": 4, "ingresos": 500000}  # Outlier
]

z_values_outliers = find_outliers(datos_outliers, "ingresos", 1)
print("z_values_outliers: ")
for z in z_values_outliers:
    print(z)

nomalization with z-score
z: -1.1619  | j: 20
z: -0.3873  | j: 30
z: +0.3873  | j: 40
z: +1.1619  | j: 50

infor about with z-score
mean: 0.0 | desv 1.0
z_values outliers
z_values_outliers: 
(3, 1.4999367987922745, {'id': 4, 'ingresos': 500000})


## Escala Logaritmica
- Para datos sesgados (pocos muy altos, muchos bajos), logaritmo comprime la escala.

In [36]:
import math


def log_scale(value: int, base=10) -> int:
    """Get the scale with a base"""
    if value <= 0:
        return None
    return math.log(value, base)


ingresos = [10000, 20000, 30000, 50000, 100000, 200000, 154800]

print(f"="*60)
print("log base 1o of ingresos")
print(f"="*60)

ingresos_normalized_b_10 = [log_scale(v) for v in ingresos]
for i, inorma in zip(ingresos, ingresos_normalized_b_10):
    print(f" ingreso: {i} -> {inorma}")
    

print("\nSummarize:")
print(f"\tRange : {min(ingresos)} - {max(ingresos)}")
print(f"\tRange log : {log_scale(min(ingresos)):.4f} - {log_scale(max(ingresos)):.4f}")
print(f"\tReduccion: de 1:100 a 1:2 (mucho mas comprimido)")




log base 1o of ingresos
 ingreso: 10000 -> 4.0
 ingreso: 20000 -> 4.30102999566398
 ingreso: 30000 -> 4.477121254719662
 ingreso: 50000 -> 4.698970004336019
 ingreso: 100000 -> 5.0
 ingreso: 200000 -> 5.301029995663981
 ingreso: 154800 -> 5.189770956346874

Summarize:
	Range : 10000 - 200000
	Range log : 4.0000 - 5.3010
	Reduccion: de 1:100 a 1:2 (mucho mas comprimido)


## CUANDO USAR LOG

In [37]:
datos_normales = [1, 2, 3, 4, 5]
datos_sesgados = [1, 2, 3, 5, 1000000]

print("\n When we must use it?")
normal_data_normalized = [log_scale(v) for v in datos_normales]
sesgados_datos_normalized = [log_scale(v) for v in datos_sesgados]

for i in range(len(datos_normales)):
    print(f"{i} | {datos_normales[i]} -> {normal_data_normalized[i]} \t\t | {datos_sesgados[i]} - {sesgados_datos_normalized[i]}")


 When we must use it?
0 | 1 -> 0.0 		 | 1 - 0.0
1 | 2 -> 0.30102999566398114 		 | 2 - 0.30102999566398114
2 | 3 -> 0.47712125471966244 		 | 3 - 0.47712125471966244
3 | 4 -> 0.6020599913279623 		 | 5 - 0.6989700043360187
4 | 5 -> 0.6989700043360187 		 | 1000000 - 5.999999999999999


## Comparations

In [56]:
import math

data = [v for v in range(10, 111, 15)]
print(f"data {data}")

print("\n[max-min]")
def get_norma_mix_min(items: list) -> list:
    """[0, 1]"""
    v_min = min(items)
    v_max = max(items)
    range = v_max - v_min
    result = []
    for i in items:
        result.append((i - v_min)/range)
    return result

norma_min_max = get_norma_mix_min(data)
for i,j in zip(data, norma_min_max):
    print(f"{i} -> {j}")
    

print("\nZ-Score")
def z_score(items:list) -> list:
    items_mean = mean(items)
    items_sta = stdev(items)
    
    if items_mean == 0:
        return[0 for _ in items]
    
    if items_sta == 0:
        return[0 for _ in items]
    
    result = []
    for i in items:
        v = (i - items_mean) / items_sta
        result.append(v)
        
    return result

norma_z_score = z_score(items=data)
for i, j in zip(data, norma_z_score):
    print(f"{i} -> {j}")


print("\nLog")
norma_log = [math.log10(v) for v in data]
for i, j in zip(data, norma_log):
    print(f"{i} -> {j}")

data [10, 25, 40, 55, 70, 85, 100]

[max-min]
10 -> 0.0
25 -> 0.16666666666666666
40 -> 0.3333333333333333
55 -> 0.5
70 -> 0.6666666666666666
85 -> 0.8333333333333334
100 -> 1.0

Z-Score
10 -> -1.3887301496588274
25 -> -0.9258200997725515
40 -> -0.46291004988627577
55 -> 0.0
70 -> 0.46291004988627577
85 -> 0.9258200997725515
100 -> 1.3887301496588274

Log
10 -> 1.0
25 -> 1.3979400086720377
40 -> 1.6020599913279623
55 -> 1.7403626894942439
70 -> 1.845098040014257
85 -> 1.9294189257142926
100 -> 2.0


## Errores Comunes

### 1. Normalizar cada columna independientemente sin guardar parámetros

| Aspecto | Detalle |
|---------|----------|
| **¿Por qué ocurre?** | Cuando aplicas el modelo a datos nuevos, no sabes qué min/max/media usaste. |
| **Solución** | Guarda parámetros en un objeto: `{edad: {min: 20, max: 50}}` y reutiliza. |

---

### 2. Normalizar el conjunto completo, luego dividir en train/test

| Aspecto | Detalle |
|---------|----------|
| **¿Por qué ocurre?** | El test set influenció la normalización. Información leakage. Resultados engañosos. |
| **Solución** | Divide PRIMERO, normaliza DENTRO de training, aplica transformación al test. |

---

### 3. Usar log(X) cuando hay valores <= 0

| Aspecto | Detalle |
|---------|----------|
| **¿Por qué ocurre?** | Log no definido para 0 o negativos. Tienes que manejar eso. |
| **Solución** | Suma constante: `log(X + 1)`, o usa Z-score en su lugar. |

---

### 4. Elegir normalización sin ver los datos

| Aspecto | Detalle |
|---------|----------|
| **¿Por qué ocurre?** | Min-Max es mala para datos sesgados (un outlier destroza la escala). |
| **Solución** | Visualiza distribución primero. ¿Simétrica? → Min-Max. ¿Sesgada? → Log o Z-score. |

---

### 5. Asumir que Z-score siempre da media=0 y std=1

| Aspecto | Detalle |
|---------|----------|
| **¿Por qué ocurre?** | Por redondeo numérico, puede estar cerca pero no exacto. |
| **Solución** | Verifica: `mean(z_scores)` y `stdev(z_scores)` después de aplicar. |